# RoomFormer Fine-Tuning on Synthetic Data

Fine-tunes the pretrained RoomFormer model on our synthetic room density maps.

**Prerequisites:**
- `training_data.zip` from your Mac (generated by `generate_training_data.py`)
- Runtime → Change runtime type → **T4 GPU**

**Steps:**
1. Upload training data
2. Clone RoomFormer + download checkpoint
3. Stub CUDA extensions + patch attention
4. Fine-tune (50 epochs, ~10-15 min on T4)
5. Export fine-tuned model as TorchScript
6. Test on synthetic validation samples
7. Download the fine-tuned `.pt` file

In [ ]:
# ── Cell 1: Upload training data ──
import os
if not os.path.exists('/content/training_data/annotations.json'):
    from google.colab import files
    print('Upload training_data.zip from your Mac:')
    uploaded = files.upload()
    !unzip -q -o training_data.zip -d /content/
    print(f'Unzipped: {len(os.listdir("/content/training_data/density/"))} density maps')
else:
    print('Training data already uploaded')

In [ ]:
# ── Cell 2: Setup environment ──
import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Clone RoomFormer
!cd /content && git clone --depth 1 https://github.com/ywyue/RoomFormer.git 2>/dev/null; echo 'repo ready'

# Install deps
!pip install -q shapely scipy opencv-python-headless pillow
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git' 2>&1 | tail -2
print('Dependencies installed')

In [ ]:
# ── Cell 3: Download pretrained checkpoint ──
import os, subprocess

CKPT_DIR = '/content/RoomFormer/checkpoints'
CKPT_PATH = f'{CKPT_DIR}/roomformer_stru3d.pth'

if not os.path.exists(CKPT_PATH):
    # Download and unzip checkpoint
    os.makedirs(CKPT_DIR, exist_ok=True)
    print('Downloading checkpoint...')
    !curl -L -o /content/ckpt.zip "https://polybox.ethz.ch/index.php/s/vlBo66X0NTrcsTC/download" 2>&1 | tail -1
    !unzip -q -o /content/ckpt.zip -d /content/RoomFormer/
    !rm /content/ckpt.zip

if os.path.exists(CKPT_PATH):
    size = os.path.getsize(CKPT_PATH) / 1e6
    print(f'Checkpoint ready: {size:.0f} MB')
else:
    print('ERROR: Checkpoint not found. Upload manually to', CKPT_PATH)
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        import shutil
        shutil.move(list(uploaded.keys())[0], CKPT_PATH)

In [ ]:
# ── Cell 4: Stub CUDA extensions + patch attention ──
import sys, types
sys.path.insert(0, '/content/RoomFormer')

# Stub compiled extensions
def _stub(*a, **k): raise RuntimeError('stub')
for mod_name in ['MultiScaleDeformableAttention', 'native_rasterizer']:
    m = types.ModuleType(mod_name)
    for attr in ['ms_deform_attn_forward', 'ms_deform_attn_backward',
                 'forward', 'backward', 'rasterize_forward', 'rasterize_backward']:
        setattr(m, attr, _stub)
    sys.modules[mod_name] = m

import torch.nn.functional as F
import numpy as np
from models.ops.functions.ms_deform_attn_func import ms_deform_attn_core_pytorch
from models.ops.modules import ms_deform_attn as attn_module

def _patched_forward(self, query, reference_points, input_flatten,
                     input_spatial_shapes, input_level_start_index,
                     input_padding_mask=None):
    N, Len_q, _ = query.shape
    N, Len_in, _ = input_flatten.shape
    value = self.value_proj(input_flatten)
    if input_padding_mask is not None:
        value = value.masked_fill(input_padding_mask[..., None], float(0))
    value = value.view(N, Len_in, self.n_heads, self.d_model // self.n_heads)
    sampling_offsets = self.sampling_offsets(query).view(
        N, Len_q, self.n_heads, self.n_levels, self.n_points, 2)
    attention_weights = self.attention_weights(query).view(
        N, Len_q, self.n_heads, self.n_levels * self.n_points)
    attention_weights = F.softmax(attention_weights, -1).view(
        N, Len_q, self.n_heads, self.n_levels, self.n_points)
    if reference_points.shape[-1] == 2:
        offset_normalizer = torch.stack(
            [input_spatial_shapes[..., 1], input_spatial_shapes[..., 0]], -1)
        sampling_locations = (
            reference_points[:, :, None, :, None, :]
            + sampling_offsets / offset_normalizer[None, None, None, :, None, :])
    elif reference_points.shape[-1] == 4:
        sampling_locations = (
            reference_points[:, :, None, :, None, :2]
            + sampling_offsets / self.n_points
            * reference_points[:, :, None, :, None, 2:] * 0.5)
    else:
        raise ValueError(f'bad dim {reference_points.shape[-1]}')
    output = ms_deform_attn_core_pytorch(
        value, input_spatial_shapes, sampling_locations, attention_weights)
    output = self.output_proj(output)
    return output

attn_module.MSDeformAttn.forward = _patched_forward
print('Stubs + patch applied')

In [ ]:
# ── Cell 5: Load model + checkpoint ──
from models import build_model
from main import get_args_parser
from util.misc import NestedTensor

parser = get_args_parser()
args = parser.parse_args([
    '--num_queries', '800', '--num_polys', '20',
    '--num_feature_levels', '4', '--backbone', 'resnet50',
    '--with_poly_refine', '--masked_attn', '--semantic_classes', '-1',
])
args.aux_loss = False

model = build_model(args, train=True)
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model'], strict=False)
print('Model loaded')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f'Using device: {device}')

In [ ]:
# ── Cell 6: Load training data ──
import json
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path

class DensityMapDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = Path(data_dir)
        with open(self.data_dir / 'annotations.json') as f:
            coco = json.load(f)
        self.images = {img['id']: img for img in coco['images']}
        self.annotations = {ann['image_id']: ann for ann in coco['annotations']}
        # Filter out degenerate samples (area < 100 px²)
        self.sample_ids = [
            sid for sid in sorted(self.images.keys())
            if self.annotations[sid]['area'] > 100
        ]
        print(f'  Loaded {len(self.sample_ids)} valid samples '
              f'(filtered {len(self.images) - len(self.sample_ids)} degenerate)')

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sid = self.sample_ids[idx]
        img_info = self.images[sid]
        ann = self.annotations[sid]

        # Load density map
        png_path = self.data_dir / 'density' / img_info['file_name']
        density = np.array(Image.open(png_path)).astype(np.float32) / 255.0

        # Parse corners
        seg = ann['segmentation'][0]
        corners = np.array(seg, dtype=np.float32).reshape(-1, 2)

        return {
            'density_map': torch.from_numpy(density).unsqueeze(0),  # [1, 256, 256]
            'corners': torch.from_numpy(corners),  # [K, 2]
            'num_corners': len(corners),
        }

dataset = DensityMapDataset('/content/training_data')
val_size = max(1, int(len(dataset) * 0.1))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))
print(f'Train: {train_size}, Val: {val_size}')

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

In [ ]:
# ── Cell 7: Fine-tune ──
import torch.optim as optim
from scipy.optimize import linear_sum_assignment

EPOCHS = 50
LR = 1e-5
FREEZE_EPOCHS = 15  # Freeze backbone for first 15 epochs

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def compute_loss(pred_coords, gt_corners_list, device):
    """L1 loss between predicted corners and ground truth via Hungarian matching."""
    total = torch.tensor(0.0, device=device, requires_grad=True)
    B = pred_coords.shape[0]
    for b in range(B):
        gt = gt_corners_list[b].to(device).float() / 255.0  # normalize to [0,1]
        K = len(gt)
        if K == 0:
            continue
        # Use room 0, first K corners
        pred = pred_coords[b, 0, :K, :]  # [K, 2]
        total = total + F.l1_loss(pred, gt)
    return total / max(B, 1)

print(f'Fine-tuning: {EPOCHS} epochs, LR={LR}')
print(f'Backbone frozen for first {FREEZE_EPOCHS} epochs')
print()

for epoch in range(EPOCHS):
    # Freeze/unfreeze backbone
    if epoch < FREEZE_EPOCHS:
        for name, p in model.named_parameters():
            if 'backbone' in name:
                p.requires_grad = False
    elif epoch == FREEZE_EPOCHS:
        for p in model.parameters():
            p.requires_grad = True
        print(f'  Epoch {epoch}: backbone unfrozen')

    model.train()
    train_loss = 0.0
    for batch in train_loader:
        density = batch['density_map'].to(device)
        masks = torch.zeros(density.shape[0], 256, 256, dtype=torch.bool, device=device)
        samples = NestedTensor(density, masks)

        outputs = model(samples)
        loss = compute_loss(outputs['pred_coords'], batch['corners'], device)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    avg_train = train_loss / max(len(train_loader), 1)

    # Validation every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                density = batch['density_map'].to(device)
                masks = torch.zeros(density.shape[0], 256, 256, dtype=torch.bool, device=device)
                outputs = model(NestedTensor(density, masks))
                loss = compute_loss(outputs['pred_coords'], batch['corners'], device)
                val_loss += loss.item()
        avg_val = val_loss / max(len(val_loader), 1)
        print(f'  Epoch {epoch+1}/{EPOCHS}: train={avg_train:.5f}, val={avg_val:.5f}')
    else:
        print(f'  Epoch {epoch+1}/{EPOCHS}: train={avg_train:.5f}')

print('\nFine-tuning complete!')

In [ ]:
# ── Cell 8: Export fine-tuned model as TorchScript ──
import torch.nn as nn

model.eval()
model.cpu()

class ExportWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, density_map):
        mask = torch.zeros(density_map.shape[0], density_map.shape[2],
                           density_map.shape[3], dtype=torch.bool,
                           device=density_map.device)
        samples = NestedTensor(density_map, mask)
        outputs = self.model(samples)
        return outputs['pred_logits'], outputs['pred_coords']

wrapper = ExportWrapper(model)
wrapper.eval()

dummy = torch.randn(1, 1, 256, 256)
with torch.no_grad():
    traced = torch.jit.trace(wrapper, (dummy,))

OUTPUT = '/content/roomformer_finetuned.pt'
traced.save(OUTPUT)
size = os.path.getsize(OUTPUT) / 1e6
print(f'Exported: {OUTPUT} ({size:.1f} MB)')

# Quick sanity check
with torch.no_grad():
    logits, coords = traced(dummy)
print(f'Output shapes: logits {logits.shape}, coords {coords.shape}')

In [ ]:
# ── Cell 9: Quick validation — does it detect rooms in synthetic data? ──
model_test = torch.jit.load(OUTPUT, map_location='cpu')
model_test.eval()

# Test on 5 validation samples
print('Validation samples (fine-tuned model):')
for i, batch in enumerate(val_loader):
    if i >= 2:  # 2 batches = ~8 samples
        break
    density = batch['density_map']
    gt_corners = batch['corners']

    with torch.no_grad():
        logits, coords = model_test(density)

    sigmoid = torch.sigmoid(logits)
    for b in range(density.shape[0]):
        gt = gt_corners[b]
        # Find best room
        best_room = -1
        best_count = 0
        for r in range(20):
            n = (sigmoid[b, r] > 0.3).sum().item()
            if n > best_count:
                best_count = n
                best_room = r

        gt_count = len(gt)
        if best_count >= 3:
            pred = coords[b, best_room, sigmoid[b, best_room] > 0.3] * 255
            # Rough area comparison
            px, py = pred[:, 0].numpy(), pred[:, 1].numpy()
            pred_area = 0.5 * abs(np.dot(px, np.roll(py, -1)) - np.dot(py, np.roll(px, -1)))
            gx, gy = gt[:, 0].numpy(), gt[:, 1].numpy()
            gt_area = 0.5 * abs(np.dot(gx, np.roll(gy, -1)) - np.dot(gy, np.roll(gx, -1)))
            area_err = abs(pred_area - gt_area) / max(gt_area, 1) * 100
            print(f'  Sample: {gt_count} GT corners → {best_count} predicted, '
                  f'area error: {area_err:.0f}%')
        else:
            print(f'  Sample: {gt_count} GT corners → no valid polygon detected')

In [ ]:
# ── Cell 10: Download fine-tuned model ──
from google.colab import files
print(f'Downloading {OUTPUT} ({os.path.getsize(OUTPUT)/1e6:.1f} MB)...')
print('Place it at: cloud/processor/models/roomformer_s3d.pt (replace the pretrained one)')
files.download(OUTPUT)